# GKX (2020) Replicatie — OLS & Elastic Net (CORRECTED)
## Financial Machine Learning Seminar

**Fixes toegepast ten opzichte van originele notebook:**
1. Recursief refit-schema: *expanding* training window + *rolling* 12-jaars validation window (zoals GKX §2.1, Internet App. D).
2. Hyperparameters worden **elk jaar opnieuw** getuned op de meegerolde validatie set (niet één keer in 1975–86 bevroren).
3. ENet grid verdicht: `alpha = logspace(-4, -1, 10)`, `l1_ratio = {0.1, 0.3, 0.5, 0.7, 0.9}`.
4. Validatiecriterium = **MSE** (zelfde loss als in training), niet R²oos.
5. OLS-Full ook in het recursive loop (was er niet eerder).
6. Huber ξ ook jaarlijks opnieuw getuned op de rolling validation.
7. Missing-value imputatie: cross-sectionele **maandmediaan per karakteristiek** (voetnoot 30 GKX).
8. OLS-3 / OLS-Full in recursive loop gebruiken alleen expanding *training* block (niet train+valid).

**Doelwaarden GKX Tabel 1 (panel R²oos, %):**
- OLS-3:    +0.16
- OLS-Full: −3.46
- ENet+H:   +0.11

**Testperiode: 1987-01 t/m 2021-12**

---
## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import warnings
from pathlib import Path
from sklearn.linear_model import LinearRegression, HuberRegressor, ElasticNet, SGDRegressor
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
np.random.seed(42)

DATA_DIR = Path('/Users/ingridmulder-kurstjens/PyCharmMiscProject/data')
OUTPUT_DIR = Path('ols_output')
OUTPUT_DIR.mkdir(exist_ok=True)

PANEL_PATH = DATA_DIR / 'gkx2020_panel.parquet'

# GKX (2020) sample splits
TRAIN_END_INIT = '1974-12'   # initial training end
VALID_END_INIT = '1986-12'   # initial validation end -> fixed 12-year window
TEST_START     = '1987-01'
TEST_END       = '2021-12'
VALID_YEARS    = 12          # rolling validation window length (fixed)
TARGET         = 'ret_excess'

# Feature definitie
pf = pq.ParquetFile(PANEL_PATH)
all_cols = pf.schema.names
NON_FEATURE_COLS = [c for c in ['permno', 'date', TARGET, 'me', 'exchcd', 'split'] if c in all_cols]
ALL_FEATURES = [c for c in all_cols if c not in NON_FEATURE_COLS]
OLS3_FEATURES = [c for c in ['mvel1', 'bm', 'mom12m'] if c in all_cols]

cols_ols3 = list(dict.fromkeys([c for c in ['date', 'permno', TARGET, 'mvel1'] + OLS3_FEATURES if c in all_cols]))
cols_full = list(dict.fromkeys([c for c in ['date', 'permno', TARGET, 'mvel1'] + ALL_FEATURES if c in all_cols]))

print(f'OLS-3 features:    {OLS3_FEATURES}')
print(f'OLS-Full features: {len(ALL_FEATURES)}')
print(f'Initial train end: {TRAIN_END_INIT}')
print(f'Initial valid end: {VALID_END_INIT}  (rolling 12-year window)')
print(f'Test period:       {TEST_START} t/m {TEST_END}')
print('Setup compleet.')

---
## 1. Hulpfuncties

In [ ]:
def r2_oos(y_true, y_pred):
    """GKX (2020) Eq. (20): denominator = sum of squared returns, no demeaning."""
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum(y_true ** 2)
    return np.nan if ss_tot == 0 else 1 - ss_res / ss_tot

def mse(y_true, y_pred):
    """Mean squared error — used as validation criterion (matches training loss)."""
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    return float(np.mean((y_true - y_pred) ** 2))

def impute_cs_median(df, feature_cols):
    """GKX footnote 30: fill missing characteristics with cross-sectional median per month."""
    df = df.copy()
    df[feature_cols] = df.groupby('date')[feature_cols].transform(
        lambda s: s.fillna(s.median())
    )
    # Any residual NaN (entire month missing) -> 0 (median of [-1,1] rank)
    df[feature_cols] = df[feature_cols].fillna(0)
    return df

def iter_split(split, cols):
    """Used only for the one-shot static models (Sections 2-4)."""
    for batch in pq.ParquetFile(PANEL_PATH).iter_batches(batch_size=200_000, columns=cols):
        df = batch.to_pandas()
        df['date'] = pd.to_datetime(df['date']).dt.to_period('M')
        ds = df['date'].astype(str)
        if split == 'train':
            mask = ds <= TRAIN_END_INIT
        elif split == 'valid':
            mask = (ds > TRAIN_END_INIT) & (ds <= VALID_END_INIT)
        elif split == 'test':
            mask = (ds > VALID_END_INIT) & (ds <= TEST_END)
        df = df[mask]
        if len(df) > 0:
            yield df

def load_split(split, cols):
    return pd.concat(list(iter_split(split, cols)), ignore_index=True)

def load_block(start_excl, end_incl, cols):
    """
    Load one contiguous time block. start_excl is exclusive (or None for 'from beginning'),
    end_incl is inclusive. Both given as 'YYYY-MM'.
    """
    parts = []
    for batch in pq.ParquetFile(PANEL_PATH).iter_batches(batch_size=200_000, columns=cols):
        df = batch.to_pandas()
        df['date'] = pd.to_datetime(df['date']).dt.to_period('M')
        ds = df['date'].astype(str)
        if start_excl is None:
            mask = ds <= end_incl
        else:
            mask = (ds > start_excl) & (ds <= end_incl)
        if mask.any():
            parts.append(df[mask])
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=cols)

def max_drawdown(returns):
    cum = (1 + pd.Series(returns)).cumprod()
    return ((cum - cum.cummax()) / cum.cummax()).min()

def collect_test_metrics(model, features, cols, label):
    ss_res, ss_tot = 0.0, 0.0
    ss_res_top, ss_tot_top = 0.0, 0.0
    ss_res_bot, ss_tot_bot = 0.0, 0.0
    annual = {}
    dec_rets = {d: [] for d in range(1, 11)}
    mse_series = {}
    ls_series = {}

    for df in iter_split('test', cols):
        df = df.dropna(subset=[TARGET])
        if len(df) == 0:
            continue
        df = impute_cs_median(df, features)
        X = df[features].values
        df['pred'] = model.predict(X)
        y = df[TARGET].values
        p = df['pred'].values

        ss_res += ((y - p)**2).sum()
        ss_tot += (y**2).sum()

        for date, grp in df.groupby('date'):
            yr = date.year
            e2 = (grp[TARGET].values - grp['pred'].values)**2
            r2g = grp[TARGET].values**2
            if yr not in annual:
                annual[yr] = [0.0, 0.0]
            annual[yr][0] += e2.sum()
            annual[yr][1] += r2g.sum()
            mse_series[date] = mse_series.get(date, 0) + e2.mean()

            if 'mvel1' in grp.columns:
                gs = grp.sort_values('mvel1', ascending=False)
                top = gs.head(1000); bot = gs.tail(1000)
                ss_res_top += ((top[TARGET].values - top['pred'].values)**2).sum()
                ss_tot_top += (top[TARGET].values**2).sum()
                ss_res_bot += ((bot[TARGET].values - bot['pred'].values)**2).sum()
                ss_tot_bot += (bot[TARGET].values**2).sum()

            if len(grp) >= 10:
                grp = grp.copy()
                grp['dec'] = pd.qcut(grp['pred'], 10, labels=False, duplicates='drop') + 1
                for d in range(1, 11):
                    sub = grp[grp['dec'] == d]
                    if len(sub) > 0:
                        dec_rets[d].append(sub[TARGET].mean())
                top_r = grp[grp['dec'] == 10][TARGET].mean()
                bot_r = grp[grp['dec'] == 1][TARGET].mean()
                ls_series[date] = top_r - bot_r

    r2     = 1 - ss_res / ss_tot
    r2_top = 1 - ss_res_top / ss_tot_top if ss_tot_top > 0 else np.nan
    r2_bot = 1 - ss_res_bot / ss_tot_bot if ss_tot_bot > 0 else np.nan

    print(f'{label} R²oos:          {r2*100:.4f}%')
    print(f'{label} R²oos top-1000: {r2_top*100:.4f}%')
    print(f'{label} R²oos bot-1000: {r2_bot*100:.4f}%')

    return {
        'r2': r2, 'r2_top': r2_top, 'r2_bot': r2_bot,
        'annual': annual, 'dec_rets': dec_rets,
        'mse_series': mse_series, 'ls_series': ls_series
    }

def decile_table(metrics, label):
    rows = []
    for d in range(1, 11):
        rets = np.array(metrics['dec_rets'][d])
        if len(rets) == 0: continue
        rows.append({
            'Decile': d,
            'Avg Return (ann.)': np.mean(rets)*12,
            'Std (ann.)':        np.std(rets)*np.sqrt(12),
            'Sharpe':            np.mean(rets)/np.std(rets)*np.sqrt(12)
        })
    df = pd.DataFrame(rows)
    ls = np.array(list(metrics['ls_series'].values()))
    ls_sharpe = np.mean(ls)/np.std(ls)*np.sqrt(12)
    ls_avg    = np.mean(ls)*12
    print(f'\n{label} — Decile portfolios:')
    print(df.to_string(index=False))
    print(f'L-S Avg: {ls_avg:.4f}  Sharpe: {ls_sharpe:.4f}')
    return df, ls_sharpe

def risk_metrics(ls_dict, label):
    ls = pd.Series(ls_dict).sort_index()
    sharpe = ls.mean()/ls.std()*np.sqrt(12)
    mdd    = max_drawdown(ls.values)
    worst  = ls.min()
    print(f'{label}: Sharpe={sharpe:.4f}  MaxDD={mdd*100:.2f}%  Worst={worst*100:.2f}%')
    return {'Model': label, 'Sharpe': sharpe, 'MaxDD': mdd, 'Worst Month': worst}

print('Hulpfuncties gedefinieerd.')

---
## 2. Hyperparameter grids (GKX conform)

In [ ]:
# Dense grid voor ElasticNet (GKX §1.3, Internet App. E)
# lambda (sklearn 'alpha') op log-schaal: 10 waardes van 1e-4 tot 1e-1
# rho (sklearn 'l1_ratio'): 5 waardes tussen 0.1 en 0.9 (pure ridge en pure lasso uitgesloten)
ALPHA_GRID    = np.logspace(-4, -1, 10)
L1_RATIO_GRID = [0.1, 0.3, 0.5, 0.7, 0.9]

# Huber xi grid (ξ in Eq. 6 van het paper)
XI_GRID = [1.35, 2.0, 5.0, 10.0, 20.0]

print(f'ENet alpha grid: {ALPHA_GRID}')
print(f'ENet l1_ratio grid: {L1_RATIO_GRID}')
print(f'Totaal grid punten per jaar: {len(ALPHA_GRID)*len(L1_RATIO_GRID)}')

---
## 3. Eénmalige baseline fits (voor snel overzicht — geen recursie)

Deze blokken trainen één keer op 1957–1974 en evalueren op 1987–2021. Ze dienen als snel sanity-check; de *echte* GKX-cijfers komen uit de recursieve loop in Sectie 5.

In [ ]:
# OLS-3 static
print('OLS-3 (static) schatten...')
train3 = load_split('train', cols_ols3).dropna(subset=OLS3_FEATURES + [TARGET])
train3 = impute_cs_median(train3, OLS3_FEATURES)
ols3 = LinearRegression().fit(train3[OLS3_FEATURES].values, train3[TARGET].values)
print(f'Geschat op {len(train3):,} obs.')
for f, c in zip(OLS3_FEATURES, ols3.coef_):
    print(f'  {f}: {c:.6f}')
del train3

print('Testset evalueren...')
m_ols3_static = collect_test_metrics(ols3, OLS3_FEATURES, cols_ols3, 'OLS-3 (static)')

In [ ]:
# OLS-Full static
print('OLS-Full (static) schatten...')
train_f = load_split('train', cols_full).dropna(subset=[TARGET])
train_f = impute_cs_median(train_f, ALL_FEATURES)
ols_full = LinearRegression().fit(train_f[ALL_FEATURES].values, train_f[TARGET].values)
print(f'Geschat op {len(train_f):,} obs.')
del train_f

print('Testset evalueren...')
m_full_static = collect_test_metrics(ols_full, ALL_FEATURES, cols_full, 'OLS-Full (static)')

---
## 4. OLS-3 + Huber (static — één keer xi tunen voor snelheid)

In [ ]:
print('OLS-3+H (static) schatten — xi tunen op initiele validatie 1975-1986...')
train3 = load_split('train', cols_ols3).dropna(subset=OLS3_FEATURES + [TARGET])
valid3 = load_split('valid', cols_ols3).dropna(subset=OLS3_FEATURES + [TARGET])
train3 = impute_cs_median(train3, OLS3_FEATURES)
valid3 = impute_cs_median(valid3, OLS3_FEATURES)

best_xi3, best_mse_v3 = None, np.inf
for xi in XI_GRID:
    hub = HuberRegressor(epsilon=xi, max_iter=300)
    hub.fit(train3[OLS3_FEATURES].values, train3[TARGET].values)
    mv = mse(valid3[TARGET].values, hub.predict(valid3[OLS3_FEATURES].values))
    print(f'  xi={xi:>5}  val MSE={mv:.6f}')
    if mv < best_mse_v3:
        best_mse_v3, best_xi3 = mv, xi

hub3 = HuberRegressor(epsilon=best_xi3, max_iter=300)
hub3.fit(train3[OLS3_FEATURES].values, train3[TARGET].values)
print(f'Beste xi OLS-3+H: {best_xi3}')
del train3, valid3

m_hub3_static = collect_test_metrics(hub3, OLS3_FEATURES, cols_ols3, 'OLS-3+H (static)')

---
## 5. HOOFDRESULTAAT — Recursief herschatten conform GKX §2.1

**Schema per jaar t ∈ {1987, …, 2021}:**
- **Train**: 1957-03 t/m `(t − 13)`-12  ← expanderend (begint op 1974)
- **Valid**: `(t − 12)`-01 t/m `(t − 1)`-12 ← rollend, lengte vast = 12 jaar
- **Test** : `t`-01 t/m `t`-12

In dit blok worden **elk jaar opnieuw** de hyperparameters getuned op de meegerolde validatie set. Dit kan 30–60 minuten duren.

In [ ]:
def year_end(y):
    return f'{y}-12'

def month_before(ym_str):
    """Return 'YYYY-MM' string one month before the given 'YYYY-MM'."""
    p = pd.Period(ym_str, freq='M') - 1
    return str(p)

rec_preds = {'ols3': [], 'ols_full': [], 'enet': [], 'enet_h': []}
sparsity_over_time = []
hp_over_time = []

print('Start recursief herschatten...\n')

for year in range(1987, 2022):
    train_end = year_end(year - 13)     # 1974, 1975, 1976, ...
    valid_end = year_end(year - 1)      # 1986, 1987, 1988, ...

    # Load training + validation + test blocks (features volledig)
    tr_f = load_block(None, train_end, cols_full).dropna(subset=[TARGET])
    va_f = load_block(train_end, valid_end, cols_full).dropna(subset=[TARGET])
    te_f = load_block(valid_end, year_end(year), cols_full).dropna(subset=[TARGET])
    if len(tr_f) == 0 or len(va_f) == 0 or len(te_f) == 0:
        print(f'  {year}: skip (geen data)'); continue

    tr_f = impute_cs_median(tr_f, ALL_FEATURES)
    va_f = impute_cs_median(va_f, ALL_FEATURES)
    te_f = impute_cs_median(te_f, ALL_FEATURES)

    X_tr = tr_f[ALL_FEATURES].values; y_tr = tr_f[TARGET].values
    X_va = va_f[ALL_FEATURES].values; y_va = va_f[TARGET].values
    X_te = te_f[ALL_FEATURES].values

    # ---------- OLS-3 (geen tuning) ----------
    ols3_y = LinearRegression().fit(
        tr_f[OLS3_FEATURES].values, y_tr)
    te_f['pred_ols3'] = ols3_y.predict(te_f[OLS3_FEATURES].values)

    # ---------- OLS-Full (geen tuning) ----------
    olsf_y = LinearRegression().fit(X_tr, y_tr)
    te_f['pred_olsf'] = olsf_y.predict(X_te)

    # ---------- ENet: volledig grid-search op rolling validation ----------
    best_mse_en, best_a, best_l1, best_en = np.inf, None, None, None
    for a in ALPHA_GRID:
        for l1 in L1_RATIO_GRID:
            m = ElasticNet(alpha=a, l1_ratio=l1, max_iter=5000,
                           fit_intercept=True).fit(X_tr, y_tr)
            mv = mse(y_va, m.predict(X_va))
            if mv < best_mse_en:
                best_mse_en, best_a, best_l1, best_en = mv, a, l1, m
    te_f['pred_enet'] = best_en.predict(X_te)

    # ---------- ENet+H (Huber via SGD) ----------
    best_mse_enh, best_a_h, best_l1_h, best_enh = np.inf, None, None, None
    for a in ALPHA_GRID:
        for l1 in L1_RATIO_GRID:
            m = SGDRegressor(loss='huber', epsilon=0.1,
                             penalty='elasticnet', alpha=a, l1_ratio=l1,
                             max_iter=1000, random_state=42).fit(X_tr, y_tr)
            mv = mse(y_va, m.predict(X_va))
            if mv < best_mse_enh:
                best_mse_enh, best_a_h, best_l1_h, best_enh = mv, a, l1, m
    te_f['pred_enet_h'] = best_enh.predict(X_te)

    # ---------- bewaar predictions + HP log ----------
    nz  = int(np.sum(best_en.coef_  != 0))
    nzh = int(np.sum(best_enh.coef_ != 0))
    sparsity_over_time.append({'year': year, 'n_nonzero_enet': nz, 'n_nonzero_enet_h': nzh})
    hp_over_time.append({
        'year': year,
        'alpha_enet': best_a,   'l1_enet': best_l1,
        'alpha_enet_h': best_a_h, 'l1_enet_h': best_l1_h
    })

    rec_preds['ols3'    ].append(te_f[['date','permno',TARGET,'pred_ols3'   ,'mvel1']].rename(columns={'pred_ols3':'pred'}))
    rec_preds['ols_full'].append(te_f[['date','permno',TARGET,'pred_olsf'   ,'mvel1']].rename(columns={'pred_olsf':'pred'}))
    rec_preds['enet'    ].append(te_f[['date','permno',TARGET,'pred_enet'   ,'mvel1']].rename(columns={'pred_enet':'pred'}))
    rec_preds['enet_h'  ].append(te_f[['date','permno',TARGET,'pred_enet_h' ,'mvel1']].rename(columns={'pred_enet_h':'pred'}))

    print(f'  {year}: train<={train_end}, val<={valid_end} | '
          f'ENet α={best_a:.1e} ρ={best_l1} nz={nz:3d} | '
          f'ENet+H α={best_a_h:.1e} ρ={best_l1_h} nz={nzh:3d}')

    del tr_f, va_f, te_f, X_tr, X_va, X_te, y_tr, y_va

# Consolideer predictions
rec3_df    = pd.concat(rec_preds['ols3'],     ignore_index=True)
rec_full_df= pd.concat(rec_preds['ols_full'], ignore_index=True)
rec_en_df  = pd.concat(rec_preds['enet'],     ignore_index=True)
rec_enh_df = pd.concat(rec_preds['enet_h'],   ignore_index=True)

hp_df   = pd.DataFrame(hp_over_time)
spar_df = pd.DataFrame(sparsity_over_time)
hp_df.to_csv(OUTPUT_DIR / 'hp_over_time.csv', index=False)
spar_df.to_csv(OUTPUT_DIR / 'sparsity_over_time.csv', index=False)
print('\nRecursieve fits klaar.')

Start recursief herschatten...

  1987: train<=1974-12, val<=1986-12 | ENet α=2.2e-03 ρ=0.5 nz= 16 | ENet+H α=2.2e-02 ρ=0.5 nz=  0
  1988: train<=1975-12, val<=1987-12 | ENet α=2.2e-03 ρ=0.3 nz= 29 | ENet+H α=1.0e-02 ρ=0.9 nz=  1
  1989: train<=1976-12, val<=1988-12 | ENet α=1.0e-03 ρ=0.7 nz= 28 | ENet+H α=1.0e-03 ρ=0.9 nz=  6
  1990: train<=1977-12, val<=1989-12 | ENet α=4.6e-03 ρ=0.1 nz= 37 | ENet+H α=2.2e-03 ρ=0.7 nz=  5
  1991: train<=1978-12, val<=1990-12 | ENet α=4.6e-03 ρ=0.1 nz= 36 | ENet+H α=4.6e-03 ρ=0.5 nz=  5
  1992: train<=1979-12, val<=1991-12 | ENet α=4.6e-03 ρ=0.1 nz= 38 | ENet+H α=4.6e-03 ρ=0.7 nz=  6
  1993: train<=1980-12, val<=1992-12 | ENet α=2.2e-03 ρ=0.3 nz= 33 | ENet+H α=2.2e-04 ρ=0.9 nz= 13
  1994: train<=1981-12, val<=1993-12 | ENet α=4.6e-04 ρ=0.7 nz= 43 | ENet+H α=4.6e-03 ρ=0.5 nz=  4
  1995: train<=1982-12, val<=1994-12 | ENet α=4.6e-04 ρ=0.5 nz= 55 | ENet+H α=1.0e-02 ρ=0.5 nz=  3
  1996: train<=1983-12, val<=1995-12 | ENet α=1.0e-03 ρ=0.3 nz= 48 | ENet+H α

---
## 6. Evaluatie van de recursieve voorspellingen

In [ ]:
def eval_rec_preds(df, label):
    """Compute panel R²oos + top/bot-1000 R²oos + annual + MSE series + decile L-S from recursive predictions."""
    y = df[TARGET].values; p = df['pred'].values
    r2_panel = r2_oos(y, p)

    # Subsamples top/bot-1000 per maand
    ss_res_top, ss_tot_top = 0.0, 0.0
    ss_res_bot, ss_tot_bot = 0.0, 0.0
    annual = {}
    dec_rets = {d: [] for d in range(1, 11)}
    mse_series = {}
    ls_series = {}

    for date, grp in df.groupby('date'):
        yr = date.year
        e2  = (grp[TARGET].values - grp['pred'].values) ** 2
        r2g = grp[TARGET].values ** 2
        annual.setdefault(yr, [0.0, 0.0])
        annual[yr][0] += e2.sum();  annual[yr][1] += r2g.sum()
        mse_series[date] = e2.mean()

        gs = grp.sort_values('mvel1', ascending=False)
        top = gs.head(1000); bot = gs.tail(1000)
        ss_res_top += ((top[TARGET].values - top['pred'].values)**2).sum()
        ss_tot_top += (top[TARGET].values**2).sum()
        ss_res_bot += ((bot[TARGET].values - bot['pred'].values)**2).sum()
        ss_tot_bot += (bot[TARGET].values**2).sum()

        if len(grp) >= 10:
            grp = grp.copy()
            grp['dec'] = pd.qcut(grp['pred'], 10, labels=False, duplicates='drop') + 1
            for d in range(1, 11):
                sub = grp[grp['dec'] == d]
                if len(sub) > 0:
                    dec_rets[d].append(sub[TARGET].mean())
            top_r = grp[grp['dec'] == 10][TARGET].mean()
            bot_r = grp[grp['dec'] == 1][TARGET].mean()
            ls_series[date] = top_r - bot_r

    r2_top = 1 - ss_res_top / ss_tot_top if ss_tot_top > 0 else np.nan
    r2_bot = 1 - ss_res_bot / ss_tot_bot if ss_tot_bot > 0 else np.nan

    print(f'{label} R²oos:          {r2_panel*100:.4f}%')
    print(f'{label} R²oos top-1000: {r2_top*100:.4f}%')
    print(f'{label} R²oos bot-1000: {r2_bot*100:.4f}%')

    return {
        'r2': r2_panel, 'r2_top': r2_top, 'r2_bot': r2_bot,
        'annual': annual, 'dec_rets': dec_rets,
        'mse_series': mse_series, 'ls_series': ls_series
    }

m_ols3   = eval_rec_preds(rec3_df,     'OLS-3    (recursief)')
m_full   = eval_rec_preds(rec_full_df, 'OLS-Full (recursief)')
m_enet   = eval_rec_preds(rec_en_df,   'ENet     (recursief)')
m_enet_h = eval_rec_preds(rec_enh_df,  'ENet+H   (recursief)')

---
## 7. Tabellen & Figuren

In [ ]:
# Tabel 1: Panel & subsample R²oos
tabel1 = pd.DataFrame([
    {'Model':'OLS-3',    'All':m_ols3['r2']*100,   'Top-1000':m_ols3['r2_top']*100,   'Bot-1000':m_ols3['r2_bot']*100},
    {'Model':'OLS-Full', 'All':m_full['r2']*100,   'Top-1000':m_full['r2_top']*100,   'Bot-1000':m_full['r2_bot']*100},
    {'Model':'ENet',     'All':m_enet['r2']*100,   'Top-1000':m_enet['r2_top']*100,   'Bot-1000':m_enet['r2_bot']*100},
    {'Model':'ENet+H',   'All':m_enet_h['r2']*100, 'Top-1000':m_enet_h['r2_top']*100, 'Bot-1000':m_enet_h['r2_bot']*100},
])
print('=== TABEL 1: R²oos (%) — recursief ===')
print(tabel1.to_string(index=False))
tabel1.to_csv(OUTPUT_DIR / 'tabel1_r2oos_recursief.csv', index=False)

In [ ]:
# Tabel 2: Jaarlijkse R²oos
years = sorted(set(m_ols3['annual'].keys()) | set(m_enet['annual'].keys()))
t2_rows = []
for yr in years:
    row = {'Year': yr}
    for label, m in [('OLS-3',m_ols3),('OLS-Full',m_full),('ENet',m_enet),('ENet+H',m_enet_h)]:
        if yr in m['annual'] and m['annual'][yr][1] > 0:
            row[label] = (1 - m['annual'][yr][0]/m['annual'][yr][1])*100
        else:
            row[label] = np.nan
    t2_rows.append(row)
tabel2 = pd.DataFrame(t2_rows)
print('=== TABEL 2: Jaarlijkse R²oos (%) ===')
print(tabel2.to_string(index=False))
tabel2.to_csv(OUTPUT_DIR / 'tabel2_annual_r2oos.csv', index=False)

fig, ax = plt.subplots(figsize=(12, 4))
for col in ['OLS-3','OLS-Full','ENet','ENet+H']:
    ax.plot(tabel2['Year'], tabel2[col], label=col, marker='o', markersize=3)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Jaar'); ax.set_ylabel('R²oos (%)')
ax.set_title('Jaarlijkse R²oos — 1987 t/m 2021')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'tabel2_annual_r2oos.png', dpi=150)
plt.show()

In [ ]:
# Tabel 3: DM-test MSE-reeksen
dm_df = pd.DataFrame({
    'mse_ols3':   m_ols3['mse_series'],
    'mse_full':   m_full['mse_series'],
    'mse_enet':   m_enet['mse_series'],
    'mse_enet_h': m_enet_h['mse_series'],
}).sort_index()

# Newey-West-achtige standaardfout met lag=12
def dm_test(d):
    d = np.asarray(d)
    n = len(d); mean = d.mean(); gamma0 = ((d-mean)**2).mean()
    lag = 12
    s = gamma0
    for k in range(1, lag+1):
        w = 1 - k/(lag+1)
        g = ((d[k:]-mean)*(d[:-k]-mean)).mean()
        s += 2*w*g
    se = np.sqrt(s/n)
    return mean/se if se > 0 else np.nan

pairs = [('ols3','full'),('ols3','enet'),('ols3','enet_h'),
         ('full','enet'),('full','enet_h'),('enet','enet_h')]
dm_results = []
for a, b in pairs:
    d = dm_df[f'mse_{a}'] - dm_df[f'mse_{b}']
    dm_results.append({'model_A': a, 'model_B': b, 'DM_stat': dm_test(d.values)})
dm_tab = pd.DataFrame(dm_results)
print('=== TABEL 3: DM-test statistieken (positief = A slechter dan B) ===')
print(dm_tab.to_string(index=False))
dm_tab.to_csv(OUTPUT_DIR / 'tabel3_dm_tests.csv', index=False)
dm_df.to_csv(OUTPUT_DIR / 'tabel3_dm_mse_series.csv')

In [ ]:
# Tabel 7: Decile portfolios
dec3,   ls3   = decile_table(m_ols3,   'OLS-3')
dec_f,  ls_f  = decile_table(m_full,   'OLS-Full')
dec_e,  ls_e  = decile_table(m_enet,   'ENet')
dec_eh, ls_eh = decile_table(m_enet_h, 'ENet+H')

dec3.to_csv(OUTPUT_DIR / 'tabel7_decile_ols3.csv', index=False)
dec_f.to_csv(OUTPUT_DIR / 'tabel7_decile_ols_full.csv', index=False)
dec_e.to_csv(OUTPUT_DIR / 'tabel7_decile_enet.csv', index=False)
dec_eh.to_csv(OUTPUT_DIR / 'tabel7_decile_enet_h.csv', index=False)

In [ ]:
# Tabel 8: Risico-gecorrigeerde prestaties
t8_rows = []
for label, m in [('OLS-3', m_ols3), ('OLS-Full', m_full), ('ENet', m_enet), ('ENet+H', m_enet_h)]:
    t8_rows.append(risk_metrics(m['ls_series'], label))
tabel8 = pd.DataFrame(t8_rows)
print('\n=== TABEL 8: Risico-gecorrigeerde prestaties ===')
print(tabel8.to_string(index=False))
tabel8.to_csv(OUTPUT_DIR / 'tabel8_risk_adjusted.csv', index=False)

In [ ]:
# Figuur 3: ENet sparsiteit + gekozen HPs over tijd
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(spar_df['year'], spar_df['n_nonzero_enet'],   marker='o', label='ENet')
axes[0].plot(spar_df['year'], spar_df['n_nonzero_enet_h'], marker='s', label='ENet+H')
axes[0].set_xlabel('Jaar'); axes[0].set_ylabel('# nonzero features')
axes[0].set_title('ENet sparsiteit over tijd')
axes[0].legend()

axes[1].semilogy(hp_df['year'], hp_df['alpha_enet'],   marker='o', label='ENet α')
axes[1].semilogy(hp_df['year'], hp_df['alpha_enet_h'], marker='s', label='ENet+H α')
axes[1].set_xlabel('Jaar'); axes[1].set_ylabel('Gekozen alpha (log)')
axes[1].set_title('Gekozen ENet alpha (λ) over tijd')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figuur3_enet_sparsity_en_hp.png', dpi=150)
plt.show()

In [ ]:
# Figuur 4: Variable importance — gebruik de LAATSTE refit (most-recent coefficients)
# We herhalen ENet op het laatste train-block om coefs te extraheren
last_year = 2021
tr_last = load_block(None, year_end(last_year-13), cols_full).dropna(subset=[TARGET])
tr_last = impute_cs_median(tr_last, ALL_FEATURES)
X_last = tr_last[ALL_FEATURES].values; y_last = tr_last[TARGET].values

# Refit op beste HPs van laatste jaar
last_hp = hp_df[hp_df['year'] == last_year].iloc[0]
enet_last = ElasticNet(alpha=last_hp['alpha_enet'], l1_ratio=last_hp['l1_enet'],
                       max_iter=5000, fit_intercept=True).fit(X_last, y_last)
ols_last = LinearRegression().fit(X_last, y_last)

feat_std = tr_last[ALL_FEATURES].std().values
del tr_last, X_last, y_last

imp_full = pd.DataFrame({'feature':ALL_FEATURES, 'importance':np.abs(ols_last.coef_)*feat_std}).sort_values('importance', ascending=False)
imp_enet = pd.DataFrame({'feature':ALL_FEATURES, 'importance':np.abs(enet_last.coef_)*feat_std}).sort_values('importance', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, imp, title in zip(axes, [imp_full.head(20), imp_enet.head(20)], ['OLS-Full Top-20', 'ENet Top-20']):
    ax.barh(imp['feature'].values[::-1], imp['importance'].values[::-1], color='steelblue')
    ax.set_title(title); ax.set_xlabel('|Coef| × Std')
plt.suptitle('Variable Importance — laatste recursieve fit', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figuur4_variable_importance.png', dpi=150)
plt.show()

imp_full.to_csv(OUTPUT_DIR / 'figuur4_importance_ols_full.csv', index=False)
imp_enet.to_csv(OUTPUT_DIR / 'figuur4_importance_enet.csv', index=False)

In [ ]:
# Tabel 4: Macro Variable Importance (laatste fit)
macro_names = ['dp','ep','bm','ntis','tbl','tms','dfy','svar']
macro_imp = {}
for macro in macro_names:
    mcols = [f for f in ALL_FEATURES if f.endswith(f'_x_{macro}_macro')]
    if mcols:
        idx = [ALL_FEATURES.index(c) for c in mcols]
        macro_imp[macro] = {
            'OLS-Full': np.abs(ols_last.coef_[idx]).sum(),
            'ENet':     np.abs(enet_last.coef_[idx]).sum()
        }
macro_df = pd.DataFrame(macro_imp).T
if macro_df.sum().sum() > 0:
    macro_df = macro_df / macro_df.sum()
print('=== TABEL 4: Macro Variable Importance (genormaliseerd) ===')
print(macro_df.to_string())
macro_df.to_csv(OUTPUT_DIR / 'tabel4_macro_importance.csv')

In [ ]:
# Eindoverzicht
print('=== EINDOVERZICHT — Recursieve resultaten ===')
print(f'Testperiode: {TEST_START} t/m {TEST_END}')
print()
for label, m, doel in [
    ('OLS-3',    m_ols3,    0.16),
    ('OLS-Full', m_full,   -3.46),
    ('ENet',     m_enet,    0.11),
    ('ENet+H',   m_enet_h,  0.11),
]:
    print(f'{label:10s}  R²oos = {m["r2"]*100:+.4f}%   (doel GKX: {doel:+.2f}%)')
print()
print(f'Alle bestanden opgeslagen in: {OUTPUT_DIR.resolve()}')